# BIOSTAT 707 Course Project — Checkpoint 3

## Locked external validation on PhysioNet sets B and C

This report evaluates the model selected at Checkpoint 2 without refitting, retuning, recalibration, or predictor selection on either validation cohort. The analysis generates and saves predictions for both cohorts before reading `Outcomes-b.txt` or `Outcomes-c.txt`. This ordering preserves set-b and set-c as external validation data and prevents outcome leakage.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
OUT = ROOT / 'output'
subprocess.run(['python', 'scripts/run_checkpoint3.py'], check=True)

## Prespecified validation protocol

The frozen object is the elastic-net logistic model trained on all 4,000 set-a admissions after nested cross-validation. Predictor construction, invalid-value handling, the >90% training-missingness screen, training medians, scaling parameters, category coding, coefficients, and hyperparameters all come from Checkpoint 2. The classification threshold is selected once from set-a out-of-fold predictions by maximizing the smaller of sensitivity and positive predictive value; it is not chosen using set-b or set-c.

Primary validation measures are AUROC, precision-recall performance, flexible calibration, calibration intercept and slope, Brier and scaled Brier scores, and decision-curve net benefit. AUROC and scalar-metric confidence intervals use 2,000 stratified bootstrap resamples; flexible calibration bands use 300 stratified bootstrap resamples. Age, sex, and ICU-type results are descriptive audits, not new model-selection criteria.

In [ ]:
protocol = pd.read_csv(OUT / 'cp3_validation_protocol.csv')
display(protocol.T.rename(columns={0: 'Value'}))

## 3.1 Applying the model to set-b and set-c

In [ ]:
quality = pd.read_csv(OUT / 'cp3_data_quality.csv')
metrics = pd.read_csv(OUT / 'cp3_validation_metrics.csv')
display(quality)
display(metrics[['set', 'n', 'deaths', 'prevalence']].round(3))
for name in ['set-b', 'set-c']:
    prediction = pd.read_csv(OUT / f'cp3_{name}_predictions.csv')
    assert len(prediction) == prediction.RecordID.nunique() == 4000
    assert prediction.predicted_risk.between(0, 1).all()
    print(f'{name}: 4,000 unique predictions; range {prediction.predicted_risk.min():.4f}–{prediction.predicted_risk.max():.4f}')

The saved set-a pipeline supplies the retained predictor list and order, median/mode imputations, missingness indicators, one-hot encoding, scaling, and elastic-net coefficients. Neither validation set is used to fit any of these components. Set-b contains 4,000 stays and 568 deaths (14.2%); set-c contains 4,000 stays and 585 deaths (14.6%).

## 3.2 Discrimination

In [ ]:
discrimination = pd.read_csv(OUT / 'cp3_discrimination.csv')
auc_comparison = pd.read_csv(OUT / 'cp3_auc_comparison.csv')
auc_difference = pd.read_csv(OUT / 'cp3_auc_difference.csv')
display(discrimination.round(3))
display(auc_comparison.round(3))
display(auc_difference.round(3))
display(Image(filename=OUT / 'figures/cp3_external_roc.png'))
display(Image(filename=OUT / 'figures/cp3_set-b_precision_recall.png'))

The elastic-net model had AUROC 0.865 (95% CI 0.850–0.880) on set-b and 0.868 (0.854–0.882) on set-c, substantially above SAPS-I (0.660 and 0.638) and SOFA (0.657 and 0.629). The Checkpoint 2 nested-CV estimate was 0.843 (fold SD 0.018), so it was not optimistic; external AUROCs were slightly higher. The set-c minus set-b AUROC difference was only 0.003 (95% bootstrap CI −0.019 to 0.025), fully compatible with chance variation.

On set-b, average precision was 0.554 compared with the death-rate baseline of 0.142. The precision-recall curve is informative because deaths are a minority: it directly shows how positive predictive value changes as sensitivity changes, whereas accuracy and even ROC displays can look favorable when most patients survive.

## 3.3 Calibration

Calibration-in-the-large is estimated with the prediction logit as an offset, while the calibration slope is estimated as its coefficient in a separate logistic model. Flexible curves use LOESS with a 95% stratified-bootstrap confidence band; histograms show where predicted risks are sparse.

In [ ]:
metrics = pd.read_csv(OUT / 'cp3_validation_metrics.csv')
intervals = pd.read_csv(OUT / 'cp3_validation_intervals.csv')
display(metrics[['set', 'brier_score', 'scaled_brier_score', 'calibration_intercept', 'calibration_slope']].round(3))
display(intervals[intervals.metric.isin(['brier_score', 'scaled_brier_score', 'calibration_intercept', 'calibration_slope'])].round(3))
display(Image(filename=OUT / 'figures/cp3_calibration.png'))

Calibration was broadly good. Intercepts were −0.079 on set-b and −0.044 on set-c, indicating slight average overprediction, while slopes were 1.161 and 1.088, indicating predictions were slightly too timid rather than too extreme. The set-b LOESS curve stayed near the diagonal, with modest underprediction through parts of the middle-risk range. Set-c showed underprediction around risks of roughly 0.10–0.40 and overprediction in the sparse high-risk tail above about 0.45; uncertainty widened where few patients had high predicted risks. Brier scores were 0.087 and 0.091, and scaled Brier scores were 0.282 and 0.270.

## 3.4 Clinical usefulness

In [ ]:
clinical_utility = pd.read_csv(OUT / 'cp3_clinical_utility.csv')
display(clinical_utility.round(3))
display(Image(filename=OUT / 'figures/cp3_decision_curve.png'))

The proposed use is to flag patients for clinician review and possible goals-of-care conversation when predicted mortality risk is at least 0.30. On set-b this flags 15.3 patients per 100, with sensitivity 0.558, PPV 0.518, and Challenge event 1 score min(sensitivity, PPV) = 0.518. From thresholds 0.05–0.50, the model had greater net benefit than SOFA, treat-all, and treat-none. Nevertheless, missing about 44% of deaths is too consequential for autonomous screening; the flag could only support, not replace, clinical review.

## 3.5 Subgroups

In [ ]:
subgroups = pd.read_csv(OUT / 'cp3_subgroup_audit.csv')
display(subgroups[['subgroup_variable', 'subgroup_level', 'n', 'deaths', 'auroc', 'calibration_intercept', 'calibration_slope']].round(3))

Sex differences were small (AUROC 0.863 in females and 0.867 in males). More meaningful heterogeneity appeared by age and ICU type: AUROC declined from 0.887 below age 65 to 0.828 at age 80 or older, and ranged from 0.827 in the medical ICU to 0.877 in the surgical ICU. Cardiac-surgery calibration-in-the-large was −0.400, while its 34 deaths make that estimate imprecise. These are descriptive comparisons across multiple subgroups; differences should not be treated as established without uncertainty intervals and independent confirmation.

## 3.6 Interpretation

In [ ]:
coefficients = pd.read_csv(OUT / 'cp3_model_coefficients.csv')
display(coefficients.head(20).round(3))
display(Image(filename=OUT / 'figures/cp2_elastic_net_coefficients.png'))

The largest standardized coefficient was lower last Glasgow Coma Scale (GCS), followed by older age; other influential terms included GCS summaries, temperature, BUN, glucose, weight, and measurement counts. Coefficients describe predictive associations conditional on many correlated inputs, not causal effects. For example, `GCS_count`, `HCT_count`, and `Urine_count` partly reflect monitoring and treatment intensity: changing how often clinicians measure a variable would not necessarily change mortality. Similarly, low GCS may reflect underlying illness or sedation, so its coefficient is not the causal effect of GCS itself.

## 3.7 Most confident mistakes

In [ ]:
mistakes = pd.read_csv(OUT / 'cp3_confident_mistakes.csv')
display(mistakes)

The five highest-risk survivors had predicted risks from 0.837 to 0.903, commonly driven upward by low GCS, older age, lactate, bilirubin, BUN, glucose, or WBC. The five lowest-risk deaths had risks from 0.014 to 0.024; several were driven downward by favorable GCS values, cardiac-surgery ICU type, younger age, weight, or measurement-frequency terms. None had more than 24% of locked predictors missing, so heavy missingness alone does not explain these cases. The pattern is more consistent with model limitations, care-process signals, and outcome-relevant events after the 48-hour observation window or otherwise absent from the data.

## 3.8 Written conclusions

**Clinical usefulness.** I would not recommend autonomous clinical deployment. I would consider prospective evaluation as a nonbinding prompt for clinician review because discrimination, calibration, and decision-curve net benefit were favorable, but sensitivity at the proposed threshold was only 0.558. Workflow burden, patient preferences, and the consequences of missed deaths require evaluation before use.

**Limitations.** These retrospective cohorts come from the same historical challenge source, so they are not a contemporary multi-system validation. Predictors cover only the first 48 hours, omit later complications and many clinical-context variables, and contain informative missingness and occasional measurement errors. Subgroup and confident-error analyses are exploratory and involve multiple comparisons.

**Leakage risk.** Leakage could enter through outcome columns, validation-derived imputation or scaling, model or threshold tuning on set-b/set-c, or examining outcomes before saving predictions. The pipeline excludes all outcome fields, uses preprocessing and coefficients fitted on set-a, derives the threshold only from set-a out-of-fold predictions, saves both validation prediction files before loading their outcomes, and makes no post-validation model changes.

**Causal interpretation.** No predictor is interpreted causally. GCS, laboratory values, and measurement counts can reflect illness severity, treatment, clinician concern, and monitoring practices; their coefficients should not be read as effects of intervening on those predictors.

## Reproducibility

Run `pixi run --locked checkpoint3` from the repository root. The task uses the locked environment, executes this notebook, rebuilds all set-b/set-c validation artifacts, and writes `output/checkpoint3.html`.

## AI use statement

During the Checkpoint 3 analysis, I used GPT/Codex as a supporting tool. After reviewing the assignment requirements and the modeling approach established in Checkpoint 2, I guided the tool in applying the frozen final model to sets B and C and evaluating its external validation performance.

GPT/Codex helped me implement the reproducible validation workflow, troubleshoot technical issues, explain statistical concepts that I did not fully understand, and calculate discrimination, calibration, classification, and subgroup metrics. It also helped me develop an initial framework and organize the different parts of the analysis in a shorter amount of time.

However, relying too much on AI can make it easier to lose track of what the analysis is actually doing and overlook important details. Some of the suggestions did not fully match what I wanted, and the reasoning was sometimes divided into many separate modules without enough connection between them. I therefore had to adjust the workflow based on my own understanding and check whether the proposed methods and interpretations were appropriate.